# Prithvi-EO-2.0-300M — v2.2 Training (Corrientes + Australia)

**Environment**: Colab A100 (GPU required)

Combined training dataset: Corrientes Argentina 2022 (subtropical grassland) +
East Gippsland Australia 2019-2020 (temperate eucalyptus forest).
Second training biome improves cross-biome generalization.

Architecture unchanged from v2.0: Prithvi-EO-2.0-300M backbone + MultiScaleNeck + FPNDecoder.

---

## Sections

| Cells | Section | Description |
|---|---|---|
| 1-2 | Setup | Install, Drive mount |
| 3-5 | Imports, Paths, Dataset | Combined loader (Corrientes + Australia) |
| 6 | Architecture | MultiScaleNeck + FPNDecoder (same as v2.0) |
| 7-8 | **Phase 1** | 25 epochs, backbone frozen |
| 9-10 | **Phase 2** | 20 epochs, last 2 backbone blocks unfrozen |
| 11-12 | Evaluation | Threshold sweep + portfolio figure |

> **Quick eval only** (skip training): run 1-6, load checkpoint, run 11-12

In [ ]:
# [1] Install dependencies
import subprocess, sys, os

needs = False
try:
    import numpy as np
    assert tuple(int(x) for x in np.__version__.split('.')[:2]) >= (2, 0)
    from terratorch.registry import BACKBONE_REGISTRY
    print(f'OK — numpy={np.__version__}, terratorch ready.')
except Exception as e:
    needs = True
    print(f'Installing ({e})...')

if needs:
    subprocess.run([sys.executable,'-m','pip','install','-q','numpy==2.0.2','--force-reinstall'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    'terratorch','einops','timm','segmentation-models-pytorch'], check=True)
    print('Restarting kernel...')
    os.kill(os.getpid(), 9)

In [ ]:
# [2] Drive mount & environment
try:
    import google.colab; IN_COLAB = True
    from google.colab import drive; drive.mount('/content/drive')
except ImportError:
    IN_COLAB = False
print(f'IN_COLAB = {IN_COLAB}')

In [ ]:
# [3] Imports
import os, random, subprocess, warnings, time
from pathlib import Path
from tqdm import tqdm

import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from terratorch.registry import BACKBONE_REGISTRY

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch={torch.__version__}  device={DEVICE}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# [4] Paths & constants
import shutil
from concurrent.futures import ThreadPoolExecutor

if IN_COLAB:
    BASE = Path('/content/drive/MyDrive/GeoAI/wildfire-spread')
else:
    BASE = Path('..').resolve()

CKPT    = BASE / 'models/best_prithvi_v22_burn_scar_wildfire.pth'
CKPT.parent.mkdir(parents=True, exist_ok=True)
RESULTS = BASE / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

# Prithvi normalization stats (B02, B03, B04, B8A, B11, B12)
BAND_IDX     = [0, 1, 2, 4, 5, 6]
PRITHVI_MEAN = np.array([0.033349, 0.057012, 0.058897, 0.232325, 0.197285, 0.119449], dtype=np.float32)
PRITHVI_STD  = np.array([0.022691, 0.026808, 0.040041, 0.077917, 0.087087, 0.072420], dtype=np.float32)
IMG_SIZE = 224
T        = (256 - IMG_SIZE) // 2
VAL_FRAC = 0.2
BATCH    = 8
FIRE_W   = 5.0

# ── Copy patches to /content/ with parallel I/O (16 workers) ─────────────────
if IN_COLAB:
    LOCAL = Path('/content/patches_v22')
    LOCAL_IMG  = LOCAL / 'images'
    LOCAL_MASK = LOCAL / 'masks_dnbr'
    LOCAL_IMG.mkdir(parents=True, exist_ok=True)
    LOCAL_MASK.mkdir(parents=True, exist_ok=True)

    CORR_IMG  = BASE / 'data/patches/images'
    CORR_MASK = BASE / 'data/patches/masks_dnbr'
    AUS_IMG   = BASE / 'data/australia/patches_sampled/images'
    AUS_MASK  = BASE / 'data/australia/patches_sampled/masks_dnbr'

    def _parallel_copy(src_dir, dst_dir, tag, kind, workers=16):
        files = list(src_dir.glob('*.npy'))
        if not files:
            print(f'  WARNING: no patches found in {src_dir}')
            return
        existing = list(dst_dir.glob(f'{tag}_*.npy'))
        if len(existing) == len(files):
            print(f'  {tag} {kind}: {len(files)} already in /content/ -- skip')
            return
        pairs = [(f, dst_dir / f'{tag}_{f.name}') for f in files
                 if not (dst_dir / f'{tag}_{f.name}').exists()]
        print(f'  Copying {len(pairs)} {tag} {kind} ({workers} workers)...')
        def _cp(args):
            s, d = args
            shutil.copy2(s, d)
        with ThreadPoolExecutor(max_workers=workers) as ex:
            list(tqdm(ex.map(_cp, pairs), total=len(pairs), desc=f'{tag} {kind}'))

    for src, mask_src, tag in [(CORR_IMG, CORR_MASK, 'corrientes'),
                               (AUS_IMG,  AUS_MASK,  'australia')]:
        _parallel_copy(src,      LOCAL_IMG,  tag, 'images')
        _parallel_copy(mask_src, LOCAL_MASK, tag, 'masks')

    IMG_DIR  = LOCAL_IMG
    MASK_DIR = LOCAL_MASK
else:
    IMG_DIR  = BASE / 'data/patches/images'
    MASK_DIR = BASE / 'data/patches/masks_dnbr'

img_paths  = sorted(IMG_DIR.glob('*.npy'))
mask_paths = sorted(MASK_DIR.glob('*.npy'))
assert len(img_paths) == len(mask_paths), f'Mismatch: {len(img_paths)} imgs vs {len(mask_paths)} masks'

n_corrientes = sum(1 for p in img_paths if p.stem.startswith('corrientes_'))
n_australia  = sum(1 for p in img_paths if p.stem.startswith('australia_'))
print(f'Corrientes patches : {n_corrientes}')
print(f'Australia patches  : {n_australia}')
print(f'Total              : {len(img_paths)}')

In [ ]:
# [5] Dataset class
class BurnScarDataset(Dataset):
    def __init__(self, img_paths, mask_paths, augment=False):
        self.imgs, self.masks, self.augment = img_paths, mask_paths, augment

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        raw  = np.load(self.imgs[idx],  mmap_mode='r').astype(np.float32)
        mask = np.load(self.masks[idx], mmap_mode='r')
        img  = (raw[BAND_IDX] / 10000.0 - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
        img  = img[:,  T:T+IMG_SIZE, T:T+IMG_SIZE]
        mask = mask[T:T+IMG_SIZE, T:T+IMG_SIZE].copy().astype(np.float32)
        img_t, mask_t = torch.from_numpy(img), torch.from_numpy(mask).unsqueeze(0)
        if self.augment:
            if torch.rand(1) > 0.5: img_t, mask_t = TF.hflip(img_t), TF.hflip(mask_t)
            if torch.rand(1) > 0.5: img_t, mask_t = TF.vflip(img_t), TF.vflip(mask_t)
            k = torch.randint(0, 4, (1,)).item()
            if k: img_t, mask_t = TF.rotate(img_t, 90*k), TF.rotate(mask_t, 90*k)
        return img_t, mask_t

# Train/val split — stratified by fire flag
print('Scanning fire flags...')
fire_flags = np.array([np.load(p, mmap_mode='r').max() > 0 for p in tqdm(mask_paths)], dtype=np.int32)
train_idx, val_idx = train_test_split(
    np.arange(len(img_paths)), test_size=VAL_FRAC, stratify=fire_flags, random_state=SEED)

val_imgs  = [img_paths[i]  for i in val_idx]
val_masks = [mask_paths[i] for i in val_idx]
w = torch.tensor([FIRE_W if f else 1.0 for f in fire_flags[train_idx]], dtype=torch.float)
sampler = WeightedRandomSampler(w, len(w), replacement=True)

train_loader = DataLoader(
    BurnScarDataset([img_paths[i] for i in train_idx], [mask_paths[i] for i in train_idx], augment=True),
    batch_size=BATCH, sampler=sampler, num_workers=4, pin_memory=True)
val_loader = DataLoader(
    BurnScarDataset(val_imgs, val_masks, augment=False),
    batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)

print(f'Train: {len(train_idx)} | Val: {len(val_idx)} | Batches/epoch: {len(train_loader)}')

In [ ]:
# [6] Architecture — MultiScaleNeck + FPNDecoder + PrithviSegModelV2 (unchanged from v2.0)
class MultiScaleNeck(nn.Module):
    def __init__(self, n_side=14, d_model=1024, fpn_ch=256):
        super().__init__()
        self.n_side = n_side
        self.lateral = nn.ModuleList([
            nn.Sequential(nn.Conv2d(d_model, fpn_ch, 1, bias=False),
                          nn.BatchNorm2d(fpn_ch), nn.GELU()) for _ in range(4)
        ])
        self.td_conv = nn.ModuleList([
            nn.Sequential(nn.Conv2d(fpn_ch, fpn_ch, 3, padding=1, bias=False),
                          nn.BatchNorm2d(fpn_ch), nn.GELU()) for _ in range(3)
        ])
    def tok2map(self, tokens):
        B, L, D = tokens.shape
        return tokens.permute(0, 2, 1).reshape(B, D, self.n_side, self.n_side)
    def forward(self, layers_out):
        feats = [self.lateral[i](self.tok2map(layers_out[idx][:, 1:, :]))
                 for i, idx in enumerate([5, 11, 17, 23])]
        out = feats[3]
        out = self.td_conv[0](feats[2] + out)
        out = self.td_conv[1](feats[1] + out)
        out = self.td_conv[2](feats[0] + out)
        return out


class FPNDecoder(nn.Module):
    def __init__(self, in_ch=256):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 256, 2, stride=2), nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256,   128, 2, stride=2), nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128,    64, 2, stride=2), nn.BatchNorm2d(64),  nn.GELU(),
            nn.ConvTranspose2d(64,     32, 2, stride=2), nn.BatchNorm2d(32),  nn.GELU(),
            nn.Conv2d(32, 1, 1),
        )
    def forward(self, x): return self.up(x)


class PrithviSegModelV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BACKBONE_REGISTRY.build('prithvi_eo_v2_300')
        self.neck     = MultiScaleNeck(n_side=14, d_model=1024, fpn_ch=256)
        self.decoder  = FPNDecoder(in_ch=256)
    def forward(self, x):
        return self.decoder(self.neck(self.backbone(x.unsqueeze(2))))


dice_loss  = smp.losses.DiceLoss(mode='binary')
focal_loss = smp.losses.FocalLoss(mode='binary')

def criterion(logits, targets):
    return dice_loss(logits, targets) + focal_loss(logits, targets)

def fire_iou(logits, target, threshold=0.5):
    pred = (logits.sigmoid() > threshold).float()
    if target.sum() == 0 and pred.sum() == 0: return None
    tp = (pred * target).sum()
    fp = (pred * (1 - target)).sum()
    fn = ((1 - pred) * target).sum()
    return (tp / (tp + fp + fn + 1e-6)).item()

print('Architecture defined.')

In [ ]:
# [7] Phase 1 setup — backbone frozen
model = PrithviSegModelV2().to(DEVICE)

for p in model.parameters(): p.requires_grad = False
for p in model.neck.parameters(): p.requires_grad = True
for p in model.decoder.parameters(): p.requires_grad = True

LR_P1     = 5e-4
NUM_EP_P1 = 25
optimizer_p1 = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR_P1, weight_decay=1e-4)
scheduler_p1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p1, T_max=NUM_EP_P1, eta_min=1e-5)
print(f'Phase 1: {NUM_EP_P1} epochs | LR={LR_P1} | backbone frozen')
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')

In [ ]:
# [8] Phase 1 training loop
best_iou   = 0.0
history_p1 = {'train_loss': [], 'val_loss': [], 'val_iou': []}

for epoch in range(1, NUM_EP_P1 + 1):
    model.train(); model.backbone.eval()
    ep_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f'P1 Epoch {epoch:02d}/{NUM_EP_P1} [train]', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer_p1.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer_p1.step()
        ep_loss += loss.item()
    scheduler_p1.step()
    history_p1['train_loss'].append(ep_loss / len(train_loader))

    model.eval()
    v_loss, v_iou_list = 0.0, []
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f'P1 Epoch {epoch:02d}/{NUM_EP_P1} [val]', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            v_loss += criterion(preds, masks).item()
            iou = fire_iou(preds, masks)
            if iou is not None: v_iou_list.append(iou)
    history_p1['val_loss'].append(v_loss / len(val_loader))
    history_p1['val_iou'].append(np.mean(v_iou_list) if v_iou_list else 0.0)

    print(f'Phase 1 Epoch {epoch:02d} | Train: {history_p1["train_loss"][-1]:.4f} | '
          f'Val: {history_p1["val_loss"][-1]:.4f} | IoU: {history_p1["val_iou"][-1]:.4f}')

    if history_p1['val_iou'][-1] > best_iou:
        best_iou = history_p1['val_iou'][-1]
        torch.save(model.state_dict(), CKPT)
        print(f'  -> Saved (IoU: {best_iou:.4f})')

print(f'\nPhase 1 complete. Best IoU: {best_iou:.4f}')

In [ ]:
# [9] Phase 2 setup — partial backbone unfreeze (last 2 blocks)
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))

for p in model.parameters(): p.requires_grad = False
for p in model.backbone.blocks[-2:].parameters(): p.requires_grad = True
for p in model.backbone.norm.parameters(): p.requires_grad = True
for p in model.neck.parameters(): p.requires_grad = True
for p in model.decoder.parameters(): p.requires_grad = True

LR_BB_P2  = 1e-5
LR_DEC_P2 = 5e-5
NUM_EP_P2 = 20

bb_params  = [p for p in model.backbone.parameters() if p.requires_grad]
dec_params = [p for p in list(model.neck.parameters()) +
                         list(model.decoder.parameters()) if p.requires_grad]
optimizer_p2 = torch.optim.AdamW([
    {'params': bb_params,  'lr': LR_BB_P2},
    {'params': dec_params, 'lr': LR_DEC_P2},
], weight_decay=1e-4)
scheduler_p2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p2, T_max=NUM_EP_P2, eta_min=1e-6)
print(f'Phase 2: {NUM_EP_P2} epochs | LR backbone={LR_BB_P2} | LR neck+decoder={LR_DEC_P2}')

In [ ]:
# [10] Phase 2 training loop
history_p2 = {'train_loss': [], 'val_loss': [], 'val_iou': []}

for epoch in range(1, NUM_EP_P2 + 1):
    model.train()
    model.backbone.eval()
    model.backbone.blocks[-2].train()
    model.backbone.blocks[-1].train()
    model.backbone.norm.train()

    ep_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f'P2 Epoch {epoch:02d}/{NUM_EP_P2} [train]', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer_p2.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer_p2.step()
        ep_loss += loss.item()
    scheduler_p2.step()
    history_p2['train_loss'].append(ep_loss / len(train_loader))

    model.eval()
    v_loss, v_iou_list = 0.0, []
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f'P2 Epoch {epoch:02d}/{NUM_EP_P2} [val]', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            v_loss += criterion(preds, masks).item()
            iou = fire_iou(preds, masks)
            if iou is not None: v_iou_list.append(iou)
    history_p2['val_loss'].append(v_loss / len(val_loader))
    history_p2['val_iou'].append(np.mean(v_iou_list) if v_iou_list else 0.0)

    print(f'Phase 2 Epoch {epoch:02d} | Train: {history_p2["train_loss"][-1]:.4f} | '
          f'Val: {history_p2["val_loss"][-1]:.4f} | IoU: {history_p2["val_iou"][-1]:.4f}')

    if history_p2['val_iou'][-1] > best_iou:
        best_iou = history_p2['val_iou'][-1]
        torch.save(model.state_dict(), CKPT)
        print(f'  -> Saved (IoU: {best_iou:.4f})')

print(f'\nTraining complete. Best IoU: {best_iou:.4f}')

In [ ]:
# [11] Threshold optimization
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()

print('Collecting probabilities...')
all_probs, all_targets = [], []
with torch.no_grad():
    for ip, mp in tqdm(zip(val_imgs, val_masks), total=len(val_imgs)):
        raw  = np.load(ip,  mmap_mode='r').astype(np.float32)
        mask = np.load(mp,  mmap_mode='r')
        img  = (raw[BAND_IDX] / 10000.0 - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
        img  = img[:, T:T+IMG_SIZE, T:T+IMG_SIZE]
        mc   = mask[T:T+IMG_SIZE, T:T+IMG_SIZE].copy()
        prob = model(torch.from_numpy(img).unsqueeze(0).to(DEVICE)).sigmoid().squeeze().cpu().numpy()
        all_probs.append(prob.ravel())
        all_targets.append(mc.ravel())

all_probs   = np.concatenate(all_probs).astype(np.float32)
all_targets = np.concatenate(all_targets).astype(np.uint8)

thresholds = np.arange(0.30, 0.85, 0.025)
sweep = []
for t in thresholds:
    preds = (all_probs > t).astype(np.int32)
    tp = int(((preds==1)&(all_targets==1)).sum())
    fp = int(((preds==1)&(all_targets==0)).sum())
    fn = int(((preds==0)&(all_targets==1)).sum())
    iou  = tp/(tp+fp+fn+1e-6)
    prec = tp/(tp+fp+1e-6)
    rec  = tp/(tp+fn+1e-6)
    f1   = 2*prec*rec/(prec+rec+1e-6)
    sweep.append((t, iou, f1, prec, rec))

sweep = np.array(sweep)
bi = sweep[:,1].argmax()
THRESHOLD_OPT = float(sweep[bi, 0])
print(f'Optimal threshold: {THRESHOLD_OPT:.3f}  IoU={sweep[bi,1]:.4f}  F1={sweep[bi,2]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(sweep[:,0], sweep[:,1], label='IoU',       color='steelblue',  lw=2)
axes[0].plot(sweep[:,0], sweep[:,2], label='F1',        color='green',      lw=2)
axes[0].plot(sweep[:,0], sweep[:,3], label='Precision', color='darkorange', lw=2, ls='--')
axes[0].plot(sweep[:,0], sweep[:,4], label='Recall',    color='crimson',    lw=2, ls='--')
axes[0].axvline(THRESHOLD_OPT, color='green', ls='--', lw=1.5, label=f'Best (t={THRESHOLD_OPT:.3f})')
axes[0].set(xlabel='Threshold', ylabel='Score', title='Metrics vs Threshold')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].plot(sweep[:,4], sweep[:,3], color='steelblue', lw=2)
axes[1].scatter([sweep[bi,4]], [sweep[bi,3]], color='green', s=80, zorder=5,
                label=f't={THRESHOLD_OPT:.3f}  F1={sweep[bi,2]:.3f}')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curve')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.suptitle('Threshold Optimization — Prithvi-EO-2.0-300M v2.2 | Corrientes+Australia val set', fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS / 'threshold_sweep_v22.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# [12] Final evaluation + portfolio figure
print('Running patch-by-patch evaluation...')
all_samples = []
tp_all = fp_all = fn_all = 0
with torch.no_grad():
    for ip, mp in tqdm(zip(val_imgs, val_masks), total=len(val_imgs)):
        raw  = np.load(ip,  mmap_mode='r').astype(np.float32)
        mask = np.load(mp,  mmap_mode='r')
        img  = (raw[BAND_IDX] / 10000.0 - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
        img  = img[:, T:T+IMG_SIZE, T:T+IMG_SIZE]
        mc   = mask[T:T+IMG_SIZE, T:T+IMG_SIZE].copy()
        prob = model(torch.from_numpy(img).unsqueeze(0).to(DEVICE)).sigmoid().squeeze().cpu().numpy()
        pred = (prob > THRESHOLD_OPT).astype(np.uint8)
        tp = int(((pred==1)&(mc==1)).sum())
        fp = int(((pred==1)&(mc==0)).sum())
        fn = int(((pred==0)&(mc==1)).sum())
        iou = tp / (tp+fp+fn+1e-6)
        tp_all += tp; fp_all += fp; fn_all += fn
        rgb = raw[[2,1,0], T:T+IMG_SIZE, T:T+IMG_SIZE]
        rgb = (rgb / (np.percentile(rgb, 98)+1e-6) * 255).clip(0, 255).astype(np.uint8).transpose(1,2,0)
        all_samples.append((iou, rgb, mc, pred, tp, fp, fn, mc.sum()/(IMG_SIZE**2)))

g_iou  = tp_all / (tp_all+fp_all+fn_all+1e-6)
g_prec = tp_all / (tp_all+fp_all+1e-6)
g_rec  = tp_all / (tp_all+fn_all+1e-6)
g_f1   = 2*g_prec*g_rec / (g_prec+g_rec+1e-6)
print(f'IoU={g_iou:.4f}  F1={g_f1:.4f}  Prec={g_prec:.4f}  Rec={g_rec:.4f}')

vis = [s for s in all_samples if 0.08 <= s[7] <= 0.60]
vis.sort(key=lambda x: x[0])
n = len(vis)
picks = [('BEST', vis[int(n*.97)]), ('MED #1', vis[int(n*.62)]),
         ('MED #2', vis[int(n*.38)]), ('WORST', vis[int(n*.03)])]

def overlay(mask, pred):
    rgba = np.zeros((*mask.shape, 4), dtype=np.float32)
    rgba[(pred==1)&(mask==1)] = [0.05, 0.82, 0.12, 0.30]
    rgba[(pred==1)&(mask==0)] = [1.00, 0.52, 0.00, 0.62]
    rgba[(pred==0)&(mask==1)] = [0.88, 0.02, 0.05, 0.62]
    return rgba

fig = plt.figure(figsize=(15, 7.5), facecolor='white')
fig.suptitle('Wildfire Burn Scar Detection from Sentinel-2 Satellite Imagery',
             fontsize=14, fontweight='bold', y=0.97)
fig.text(0.5, 0.92,
         'PyTorch  |  Prithvi-EO-2.0-300M (IBM/NASA)  |  6-channel Sentinel-2 L2A  |  '
         'dNBR labels  |  Corrientes (Argentina) + East Gippsland (Australia)',
         ha='center', fontsize=8.5, color='#444')
gs   = gridspec.GridSpec(1, 2, figure=fig, left=0.02, right=0.97, top=0.89, bottom=0.15,
                         wspace=0.05, width_ratios=[2.1, 1])
gs_p = gridspec.GridSpecFromSubplotSpec(2, 2, subplot_spec=gs[0], hspace=0.03, wspace=0.03)
lcol = {'BEST':'#097a27', 'MED #1':'#b86e00', 'MED #2':'#b86e00', 'WORST':'#a30000'}
for (label, s), (r, c) in zip(picks, [(0,0),(0,1),(1,0),(1,1)]):
    iou, rgb, mc, pred = s[0], s[1], s[2], s[3]
    ax = fig.add_subplot(gs_p[r, c])
    ax.imshow(rgb, interpolation='bilinear')
    ax.imshow(overlay(mc, pred), interpolation='none')
    ax.text(0.012, 0.985, f'{label}  |  Patch IoU = {iou:.3f}', transform=ax.transAxes,
            va='top', ha='left', fontsize=8.5, fontweight='bold', color='white',
            bbox=dict(facecolor=lcol[label], edgecolor='none', pad=2.5, alpha=0.88))
    ax.axis('off')

gs_r = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs[1], hspace=0.44)
ax_b = fig.add_subplot(gs_r[0])
ep_all   = list(range(1, len(history_p1['val_iou']) + len(history_p2['val_iou']) + 1))
iou_all  = history_p1['val_iou']  + history_p2['val_iou']
loss_all = history_p1['val_loss'] + history_p2['val_loss']
ax_b.plot(ep_all, loss_all, color='steelblue', lw=1.5, label='Val loss')
ax_b.plot(ep_all, iou_all,  color='#1aaa44',   lw=1.5, label='Val IoU')
ax_b.axvline(len(history_p1['val_iou']), color='#888', ls=':', lw=1.2, label='Phase 1->2')
ax_b.set_title('Training convergence (45 epochs, Colab A100)', fontsize=8.5, fontweight='bold')
ax_b.set_xlabel('Epoch', fontsize=7.5); ax_b.tick_params(labelsize=7)
ax_b.grid(alpha=0.2); ax_b.legend(fontsize=7, frameon=False); ax_b.set_ylim(bottom=0.2)

ax_m = fig.add_subplot(gs_r[1])
names  = ['Recall', 'Precision', 'F1', 'IoU']
values = [g_rec,    g_prec,      g_f1, g_iou]
bars   = ax_m.barh(names, values, color='#4a7fbf', height=0.48)
for bar, val in zip(bars, values):
    ax_m.text(val+0.012, bar.get_y()+bar.get_height()/2, f'{val:.3f}',
              va='center', fontsize=9.5, fontweight='bold', color='#111')
ax_m.set_xlim(0, 1.05)
ax_m.set_title(f'Global validation metrics  (n={len(val_imgs):,} patches)', fontsize=8.5, fontweight='bold')
ax_m.tick_params(labelsize=8.5); ax_m.grid(axis='x', alpha=0.2)
ax_m.spines[['top','right','left']].set_visible(False)

fig.legend(handles=[
    mpatches.Patch(color=(0.05,0.82,0.12), label='True Positive'),
    mpatches.Patch(color=(1.00,0.52,0.00), label='False Positive'),
    mpatches.Patch(color=(0.88,0.02,0.05), label='False Negative'),
], loc='lower left', ncol=3, fontsize=8, frameon=False, bbox_to_anchor=(0.02, 0.075))

fig.text(0.5, 0.01,
         'Prithvi-EO-2.0-300M (IBM/NASA)  |  Multi-scale neck (layers 5,11,17,23)  |  '
         'dNBR > 0.10 (Corrientes) / dNBR > 0.15 (Australia)\n'
         'Corrientes 2022 (~900,000 ha)  +  East Gippsland 2019-2020 (~1,500,000 ha)',
         ha='center', fontsize=7, color='#555', linespacing=1.5)

plt.savefig(RESULTS / 'validation_overview_v22.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {RESULTS}/validation_overview_v22.png')

## Zero-Shot Inference -- Chile (California / Cerrado diferidos)

In [ ]:
# [14] Extra dependencies for ZS vector pipeline
import subprocess, sys
try:
    import rasterio, geopandas, pystac_client, planetary_computer, shapely, pyproj
    print('All ZS deps ready.')
except ImportError as e:
    print(f'Installing ({e})...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'rasterio', 'geopandas', 'pystac-client',
                    'planetary-computer', 'shapely', 'pyproj'], check=True)
    print('Done -- re-run this cell to verify.')

In [ ]:
# [15] ZS imports, paths and site config
import json, shutil
import rasterio
from rasterio.features import shapes
from rasterio.transform import from_origin
import geopandas as gpd
from shapely.geometry import shape
from shapely.ops import unary_union
import pystac_client
import planetary_computer
from pyproj import Transformer
from torch.utils.data import Dataset, DataLoader

RES         = 10    # Sentinel-2 resolution (m)
MIN_AREA_HA = 10    # minimum polygon to keep after rasterization
ZS_THRESHOLD = globals().get('THRESHOLD_OPT', 0.45)
print(f'ZS threshold: {ZS_THRESHOLD:.3f}')

OUTPUTS_V22 = BASE / 'outputs' / 'v2.2'
OUTPUTS_V22.mkdir(parents=True, exist_ok=True)

SITES = {
    'chile': {
        'label':    'Chile ZS #1 (Mediterranean WUI)',
        'data_dir': BASE / 'data' / 'zs' / 'chile' / 'patches',
        'color':    '#E74C3C',
        'date':     '2023-02',
    },
    'california': {
        'label':    'California ZS #2 (Temperate conifer, Sierra Nevada)',
        'data_dir': BASE / 'data' / 'zs' / 'california' / 'patches',
        'color':    '#E67E22',
        'date':     '2021-08',
    },
    'cerrado': {
        'label':    'Cerrado ZS #3 (Tropical dry savanna)',
        'data_dir': BASE / 'data' / 'zs' / 'cerrado' / 'patches',
        'color':    '#27AE60',
        'date':     '2023-08',
    },
}
print('Sites:', list(SITES.keys()))

In [ ]:
# [16] PatchDataset + batch inference
class PatchDataset(Dataset):
    def __init__(self, img_paths):
        self.img_paths = img_paths
    def __len__(self): return len(self.img_paths)
    def __getitem__(self, idx):
        arr = np.load(self.img_paths[idx], mmap_mode='r').astype(np.float32)
        img = arr[BAND_IDX] / 10000.0
        img = (img - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
        img = img[:, T:T+IMG_SIZE, T:T+IMG_SIZE]
        return torch.from_numpy(img)


def run_inference(img_paths, batch_size=8, num_workers=2):
    ds     = PatchDataset(img_paths)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)
    probs_list = []
    with torch.no_grad():
        for imgs in tqdm(loader, desc='Inference'):
            probs = model(imgs.to(DEVICE)).sigmoid().squeeze(1).cpu().numpy()
            for b in range(len(imgs)):
                probs_list.append(probs[b])
    return probs_list  # list of (IMG_SIZE, IMG_SIZE) float32


print('Inference helpers defined.')

In [ ]:
# [17] Scene reconstruction with NaN for no-data
def parse_row_col(path):
    stem  = Path(path).stem
    parts = stem.split('_r')
    rc    = parts[-1]
    return int(rc.split('_c')[0]), int(rc.split('_c')[1])


def parse_tile_id(path):
    stem  = Path(path).stem
    parts = stem.split('_r')[0].split('_')
    for p in reversed(parts):
        if len(p) >= 5 and p[-2:].isalpha():
            return p.lstrip('T')
    return None


def reconstruct_scene(img_paths, probs_list, patch_size=IMG_SIZE, crop_offset=T):
    rows_cols = []
    for p in img_paths:
        r, c = parse_row_col(p)
        rows_cols.append((r + crop_offset, c + crop_offset))

    max_row = max(r for r, c in rows_cols) + patch_size
    max_col = max(c for r, c in rows_cols) + patch_size

    scene_sum   = np.zeros((max_row, max_col), dtype=np.float32)
    scene_count = np.zeros((max_row, max_col), dtype=np.float32)

    for (row, col), prob in zip(rows_cols, probs_list):
        scene_sum  [row:row+patch_size, col:col+patch_size] += prob
        scene_count[row:row+patch_size, col:col+patch_size] += 1

    has_data   = scene_count > 0
    scene_prob = np.where(has_data, scene_sum / np.maximum(scene_count, 1), np.nan)
    return scene_prob, max_row, max_col, rows_cols


print('Reconstruction helpers defined (NaN masking).')

In [ ]:
# [18] Geographic coordinate recovery -- STAC + pyproj fallback
_tile_cache = {}

def get_tile_origin(mgrs_tile_no_t):
    if mgrs_tile_no_t in _tile_cache:
        return _tile_cache[mgrs_tile_no_t]

    zone = int(mgrs_tile_no_t[:2])
    epsg = 32600 + zone  # UTM N; corrected below if S

    try:
        catalog = pystac_client.Client.open(
            'https://planetarycomputer.microsoft.com/api/stac/v1',
            modifier=planetary_computer.sign_inplace,
        )
        search = catalog.search(
            collections=['sentinel-2-l2a'],
            query={'s2:mgrs_tile': {'eq': mgrs_tile_no_t}},
            max_items=1,
        )
        items = list(search.items())
        if not items:
            raise ValueError(f'Tile {mgrs_tile_no_t} not found in STAC')

        item      = items[0]
        proj_bbox = item.properties.get('proj:bbox')
        proj_epsg = item.properties.get('proj:epsg')

        if proj_bbox and proj_epsg:
            ox, oy = float(proj_bbox[0]), float(proj_bbox[3])
            epsg   = int(proj_epsg)
            print(f'  STAC proj:bbox {mgrs_tile_no_t}: ({ox:.0f}, {oy:.0f}) EPSG:{epsg}')
        else:
            t = Transformer.from_crs('EPSG:4326', f'EPSG:{epsg}', always_xy=True)
            mn_lon, mn_lat, mx_lon, mx_lat = item.bbox
            ox, oy = t.transform(mn_lon, mx_lat)
            print(f'  STAC bbox->UTM {mgrs_tile_no_t}: ({ox:.0f}, {oy:.0f}) EPSG:{epsg}')

    except Exception as e:
        print(f'  STAC failed for {mgrs_tile_no_t}: {e}')
        print('  WARNING: geographic extent will be in arbitrary pixel coordinates')
        ox, oy = 0.0, 0.0

    result = (ox, oy, epsg)
    _tile_cache[mgrs_tile_no_t] = result
    return result


print('Geographic recovery defined (STAC + pyproj fallback).')

In [ ]:
# [19] Vectorization: probability map -> GeoDataFrame polygons
def vectorize_scene(scene_prob, tile_origin_x, tile_origin_y, epsg,
                    scene_row_offset=0, scene_col_offset=0,
                    threshold=None, res=RES,
                    min_area_ha=MIN_AREA_HA, dissolve_area_ha=100,
                    site_name='', event_date=''):
    if threshold is None:
        threshold = ZS_THRESHOLD

    valid_mask = ~np.isnan(scene_prob)
    binary     = np.zeros_like(scene_prob, dtype=np.uint8)
    binary[valid_mask & (scene_prob >= threshold)] = 1

    west      = tile_origin_x + scene_col_offset * res
    north     = tile_origin_y - scene_row_offset * res
    transform = from_origin(west, north, res, res)

    geoms = [
        shape(geom)
        for geom, val in shapes(binary, mask=binary, transform=transform)
        if val == 1
    ]

    if not geoms:
        print(f'  No polygons above threshold {threshold:.3f} for {site_name}')
        return gpd.GeoDataFrame(
            columns=['geometry','area_ha','perimeter_km','site','date','model'],
            crs=f'EPSG:{epsg}')

    gdf = gpd.GeoDataFrame(geometry=geoms, crs=f'EPSG:{epsg}')
    gdf['area_ha'] = gdf.geometry.area / 10000
    gdf = gdf[gdf['area_ha'] >= min_area_ha].reset_index(drop=True)
    print(f'  After {min_area_ha} ha filter: {len(gdf)} polygons')

    if len(gdf) > 0:
        buf       = T * res   # 16 px * 10 m = 160 m -- bridges center-crop gaps
        dissolved = unary_union(gdf.geometry.buffer(buf))
        parts     = list(dissolved.geoms) if hasattr(dissolved, 'geoms') else [dissolved]
        gdf       = gpd.GeoDataFrame(geometry=parts, crs=f'EPSG:{epsg}')
        gdf['area_ha']      = gdf.geometry.area / 10000
        gdf['perimeter_km'] = gdf.geometry.length / 1000
        gdf = gdf[gdf['area_ha'] >= dissolve_area_ha].reset_index(drop=True)
        print(f'  After dissolve + {dissolve_area_ha} ha: {len(gdf)} polygons')

    gdf['site']  = site_name
    gdf['date']  = event_date
    gdf['model'] = 'prithvi_eo_v2_300_v22_zs'
    print(f'  Total area: {gdf["area_ha"].sum():.0f} ha')
    return gdf


print('Vectorization defined (buf=160m, no shrink).')

In [ ]:
# [20] Figure helpers
def norm_band(x, p=2):
    v = x[np.isfinite(x) & (x > 0)]
    if not len(v): return np.zeros_like(x)
    lo, hi = np.percentile(v, [p, 100 - p])
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)


def save_fig(fig, name, drive_dir=None, dpi=150):
    if drive_dir is None:
        drive_dir = RESULTS
    if IN_COLAB:
        local_path = Path('/content') / name
        fig.savefig(local_path, dpi=dpi, bbox_inches='tight')
        plt.show()
        drive_dir.mkdir(parents=True, exist_ok=True)
        dst = drive_dir / name
        shutil.copy(local_path, dst)
        assert dst.stat().st_size > 1000, f'Drive copy too small: {dst}'
        print(f'Saved: {dst} ({dst.stat().st_size // 1024} KB)')
    else:
        drive_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(drive_dir / name, dpi=dpi, bbox_inches='tight')
        plt.show()
        print(f'Saved: {drive_dir / name}')


def reconstruct_rgb_scene(img_paths, patch_size=IMG_SIZE, crop_offset=T):
    rows_cols = []
    for p in img_paths:
        r, c = parse_row_col(p)
        rows_cols.append((r + crop_offset, c + crop_offset))

    max_row = max(r for r, c in rows_cols) + patch_size
    max_col = max(c for r, c in rows_cols) + patch_size

    scene_r = np.zeros((max_row, max_col), dtype=np.float32)
    scene_g = np.zeros((max_row, max_col), dtype=np.float32)
    scene_b = np.zeros((max_row, max_col), dtype=np.float32)
    cnt     = np.zeros((max_row, max_col), dtype=np.float32)

    for (row, col), p in zip(rows_cols, img_paths):
        arr  = np.load(p, mmap_mode='r').astype(np.float32)
        crop = slice(crop_offset, crop_offset + patch_size)
        scene_r[row:row+patch_size, col:col+patch_size] += arr[2, crop, crop]
        scene_g[row:row+patch_size, col:col+patch_size] += arr[1, crop, crop]
        scene_b[row:row+patch_size, col:col+patch_size] += arr[0, crop, crop]
        cnt    [row:row+patch_size, col:col+patch_size] += 1

    c   = np.maximum(cnt, 1)
    rgb = np.stack([norm_band(scene_r / c),
                    norm_band(scene_g / c),
                    norm_band(scene_b / c)], axis=-1)
    rgb[cnt == 0] = 0
    return rgb


def plot_vector_overview(rgb_scene, prob_scene, gdf, site_label, site_color):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), facecolor='white')

    axes[0].imshow(rgb_scene)
    axes[0].set_title('RGB mosaic (Sentinel-2 post-fire)', fontsize=9)
    axes[0].axis('off')

    im = axes[1].imshow(prob_scene, cmap='RdYlGn_r', vmin=0, vmax=1)
    plt.colorbar(im, ax=axes[1], fraction=0.04, pad=0.02, label='P(burned)')
    axes[1].set_title('Burn probability (v2.2)', fontsize=9)
    axes[1].axis('off')

    if len(gdf) > 0:
        total_ha = gdf['area_ha'].sum()
        gdf.plot(ax=axes[2], color=site_color, alpha=0.7,
                 edgecolor='white', linewidth=0.3)
        axes[2].set_title(f'Burn scar polygons\n{len(gdf)} features -- {total_ha:.0f} ha', fontsize=9)
        axes[2].set_xlabel('UTM Easting (m)', fontsize=7)
        axes[2].set_ylabel('UTM Northing (m)', fontsize=7)
        axes[2].tick_params(labelsize=6)
        for sp in axes[2].spines.values(): sp.set_color(site_color)
    else:
        axes[2].set_title('No polygons detected', fontsize=9)
        axes[2].axis('off')

    plt.suptitle(f'v2.2 Zero-Shot Inference -- {site_label}', fontsize=11, fontweight='bold')
    plt.tight_layout()
    return fig


print('Figure helpers defined.')

In [ ]:
# [21] Cordoba ZS -- fase anterior (Cordoba/Greece/Canada), CERRADA el 2026-07-07.
# No se usa en el showcase final v2.2 (Chile/California/Cerrado). Resultados ya guardados en outputs/v2.2/.
# Codigo comentado como referencia historica.
"""
# [21] Cordoba ZS #1 (Argentine Monte)
SITE = 'cordoba'
cfg  = SITES[SITE]
print(f'Processing: {cfg["label"]}')
print('=' * 60)

# Reload best v2.2 checkpoint
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()
print(f'Model reloaded from {CKPT.name}')

img_dir = cfg['data_dir'] / 'images'
if IN_COLAB:
    local_co = Path('/content/cordoba_patches')
    if not (local_co / 'images').exists():
        print('Copying patches from Drive...')
        local_co.mkdir(exist_ok=True)
        for sub in ('images', 'masks_dnbr'):
            src = cfg['data_dir'] / sub
            if not src.exists():
                print(f'  (skip {sub}, no existe)')
                continue
            dst = local_co / sub
            for attempt in range(3):
                result = subprocess.run(['rsync', '-a', f'{src}/', f'{dst}/'])
                if result.returncode == 0:
                    break
                print(f'  intento {attempt+1} fallo (exit {result.returncode}), reintentando...')
            else:
                raise RuntimeError(f'No se pudo copiar {sub} tras 3 intentos')
    img_dir = local_co / 'images'

co_img_paths = sorted(img_dir.glob('*.npy'))
print(f'Patches: {len(co_img_paths)}')
assert co_img_paths, f'No patches found in {img_dir}'

co_probs = run_inference(co_img_paths)
co_scene, _, _, _ = reconstruct_scene(co_img_paths, co_probs)
print(f'Scene shape: {co_scene.shape}')

co_tile = parse_tile_id(co_img_paths[0])
print(f'Tile ID: {co_tile}')
co_ox, co_oy, co_epsg = get_tile_origin(co_tile) if co_tile else (0.0, 0.0, 32720)

co_gdf = vectorize_scene(co_scene, co_ox, co_oy, co_epsg,
                         site_name='cordoba', event_date=cfg['date'])

if len(co_gdf) > 0:
    gpkg_local = Path('/content/cordoba_burn_scar_v22.gpkg') if IN_COLAB else OUTPUTS_V22 / 'cordoba_burn_scar_v22.gpkg'
    co_gdf.to_file(gpkg_local, driver='GPKG')
    if IN_COLAB:
        shutil.copy(gpkg_local, OUTPUTS_V22 / 'cordoba_burn_scar_v22.gpkg')
    print(f'GeoPackage: {OUTPUTS_V22 / "cordoba_burn_scar_v22.gpkg"}')
else:
    print('No polygons -- GeoPackage skipped')

co_rgb = reconstruct_rgb_scene(co_img_paths)
fig_co = plot_vector_overview(co_rgb, co_scene, co_gdf, cfg['label'], cfg['color'])
save_fig(fig_co, 'cordoba_vector_output_v22.png')
print('Cordoba DONE.')
"""

In [ ]:
# [22] Greece ZS -- fase anterior (Cordoba/Greece/Canada), CERRADA el 2026-07-07.
# No se usa en el showcase final v2.2 (Chile/California/Cerrado). Resultados ya guardados en outputs/v2.2/.
# Codigo comentado como referencia historica.
"""
# [22] Greece ZS #2 (Mediterranean shrubland)
SITE = 'greece'
cfg  = SITES[SITE]
print(f'Processing: {cfg["label"]}')
print('=' * 60)

img_dir = cfg['data_dir'] / 'images'
if IN_COLAB:
    local_gr = Path('/content/greece_patches')
    if not (local_gr / 'images').exists():
        print('Copying patches from Drive...')
        local_gr.mkdir(exist_ok=True)
        for sub in ('images', 'masks_dnbr'):
            src = cfg['data_dir'] / sub
            if not src.exists():
                print(f'  (skip {sub}, no existe)')
                continue
            dst = local_gr / sub
            for attempt in range(3):
                result = subprocess.run(['rsync', '-a', f'{src}/', f'{dst}/'])
                if result.returncode == 0:
                    break
                print(f'  intento {attempt+1} fallo (exit {result.returncode}), reintentando...')
            else:
                raise RuntimeError(f'No se pudo copiar {sub} tras 3 intentos')
    img_dir = local_gr / 'images'

gr_img_paths = sorted(img_dir.glob('*.npy'))
print(f'Patches: {len(gr_img_paths)}')
assert gr_img_paths, f'No patches found in {img_dir}'

# Select most-burned tile via manifest
manifest_path = cfg['data_dir'] / 'manifest.json'
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    best_tile = max(manifest['tiles'], key=lambda t: t['burned_pct'])['tile'].lstrip('T')
    print(f'Most burned tile: {best_tile}')
    gr_img_tile = [p for p in gr_img_paths if best_tile in p.stem] or gr_img_paths
else:
    gr_img_tile = gr_img_paths
    best_tile   = parse_tile_id(gr_img_tile[0]) or 'unknown'
    print(f'No manifest -- tile: {best_tile}')

gr_probs_all  = run_inference(gr_img_paths)
tile_indices  = [i for i, p in enumerate(gr_img_paths) if best_tile in p.stem]
gr_probs_tile = [gr_probs_all[i] for i in tile_indices] if tile_indices else gr_probs_all

gr_scene, _, _, _ = reconstruct_scene(gr_img_tile, gr_probs_tile)
print(f'Scene shape: {gr_scene.shape}')

gr_ox, gr_oy, gr_epsg = get_tile_origin(best_tile) if best_tile != 'unknown' else (0.0, 0.0, 32635)

gr_gdf = vectorize_scene(gr_scene, gr_ox, gr_oy, gr_epsg,
                         site_name='greece', event_date=cfg['date'])

if len(gr_gdf) > 0:
    gpkg_local = Path('/content/greece_burn_scar_v22.gpkg') if IN_COLAB else OUTPUTS_V22 / 'greece_burn_scar_v22.gpkg'
    gr_gdf.to_file(gpkg_local, driver='GPKG')
    if IN_COLAB:
        shutil.copy(gpkg_local, OUTPUTS_V22 / 'greece_burn_scar_v22.gpkg')
    print(f'GeoPackage: {OUTPUTS_V22 / "greece_burn_scar_v22.gpkg"}')
else:
    print('No polygons -- GeoPackage skipped')

gr_rgb = reconstruct_rgb_scene(gr_img_tile)
fig_gr = plot_vector_overview(gr_rgb, gr_scene, gr_gdf, cfg['label'], cfg['color'])
save_fig(fig_gr, 'greece_vector_output_v22.png')
print('Greece DONE.')
"""

In [ ]:
# [23] Canada ZS -- fase anterior (Cordoba/Greece/Canada), CERRADA el 2026-07-07.
# No se usa en el showcase final v2.2 (Chile/California/Cerrado). Resultados ya guardados en outputs/v2.2/.
# Codigo comentado como referencia historica.
"""
# [23] Canada ZS #3 (Boreal forest)
SITE = 'canada'
cfg  = SITES[SITE]
print(f'Processing: {cfg["label"]}')
print('=' * 60)

img_dir = cfg['data_dir'] / 'images'
if IN_COLAB:
    local_ca = Path('/content/canada_patches')
    if not (local_ca / 'images').exists():
        print('Copying patches from Drive...')
        local_ca.mkdir(exist_ok=True)
        for sub in ('images', 'masks_dnbr'):
            src = cfg['data_dir'] / sub
            if not src.exists():
                print(f'  (skip {sub}, no existe)')
                continue
            dst = local_ca / sub
            for attempt in range(3):
                result = subprocess.run(['rsync', '-a', f'{src}/', f'{dst}/'])
                if result.returncode == 0:
                    break
                print(f'  intento {attempt+1} fallo (exit {result.returncode}), reintentando...')
            else:
                raise RuntimeError(f'No se pudo copiar {sub} tras 3 intentos')
    img_dir = local_ca / 'images'

ca_img_paths_all = sorted(img_dir.glob('*.npy'))
print(f'Total patches: {len(ca_img_paths_all)}')
assert ca_img_paths_all, f'No patches found in {img_dir}'

# Filter cloudy / water patches if quality bands are present (shape > 10)
MIN_VALID_FRAC = 0.70
MAX_WATER_FRAC = 0.30
keep = []
sample = np.load(ca_img_paths_all[0], mmap_mode='r')
has_quality_bands = sample.shape[0] > 10
if has_quality_bands:
    for i, ip in enumerate(tqdm(ca_img_paths_all, desc='Quality filter')):
        arr = np.load(ip, mmap_mode='r')
        if arr[10].astype(np.float32).mean() < MIN_VALID_FRAC: continue
        if (arr[9].astype(np.float32) / 10000.0 > 0.1).mean() > MAX_WATER_FRAC: continue
        keep.append(i)
    ca_img_paths = [ca_img_paths_all[i] for i in keep]
    print(f'After quality filter: {len(ca_img_paths)} patches')
else:
    ca_img_paths = ca_img_paths_all
    print('No quality bands -- using all patches')

# Select most-burned tile
manifest_path = cfg['data_dir'] / 'manifest.json'
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    ca_main_tile = max(manifest['tiles'], key=lambda t: t['burned_pct'])['tile'].lstrip('T')
    print(f'Most burned tile: {ca_main_tile}')
else:
    ca_main_tile = parse_tile_id(ca_img_paths[0]) or 'unknown'
    print(f'No manifest -- tile: {ca_main_tile}')

ca_img_tile   = [p for p in ca_img_paths if ca_main_tile in p.stem] or ca_img_paths
ca_probs_all  = run_inference(ca_img_paths)
tile_indices  = [i for i, p in enumerate(ca_img_paths) if ca_main_tile in p.stem]
ca_probs_tile = [ca_probs_all[i] for i in tile_indices] if tile_indices else ca_probs_all

ca_scene, _, _, _ = reconstruct_scene(ca_img_tile, ca_probs_tile)
print(f'Scene shape: {ca_scene.shape}')

ca_ox, ca_oy, ca_epsg = get_tile_origin(ca_main_tile) if ca_main_tile != 'unknown' else (0.0, 0.0, 32611)

ca_gdf = vectorize_scene(ca_scene, ca_ox, ca_oy, ca_epsg,
                         site_name='canada', event_date=cfg['date'])

if len(ca_gdf) > 0:
    gpkg_local = Path('/content/canada_burn_scar_v22.gpkg') if IN_COLAB else OUTPUTS_V22 / 'canada_burn_scar_v22.gpkg'
    ca_gdf.to_file(gpkg_local, driver='GPKG')
    if IN_COLAB:
        shutil.copy(gpkg_local, OUTPUTS_V22 / 'canada_burn_scar_v22.gpkg')
    print(f'GeoPackage: {OUTPUTS_V22 / "canada_burn_scar_v22.gpkg"}')
else:
    print('No polygons -- GeoPackage skipped')

ca_rgb = reconstruct_rgb_scene(ca_img_tile)
fig_ca = plot_vector_overview(ca_rgb, ca_scene, ca_gdf, cfg['label'], cfg['color'])
save_fig(fig_ca, 'canada_vector_output_v22.png')
print('Canada DONE.')
"""

In [ ]:
# [23a] Chile ZS #1 (Mediterranean WUI) -- Valparaiso wildfire 2023
SITE = 'chile'
cfg  = SITES[SITE]
print(f'Processing: {cfg["label"]}')
print('=' * 60)

# Reload best v2.2 checkpoint
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()
print(f'Model reloaded from {CKPT.name}')

img_dir = cfg['data_dir']
if IN_COLAB:
    local_ch = Path('/content/chile_patches')
    if not any(local_ch.glob('*.npy')):
        print('Copying patches from Drive...')
        local_ch.mkdir(exist_ok=True)
        for attempt in range(3):
            result = subprocess.run(['rsync', '-a', f'{cfg["data_dir"]}/', f'{local_ch}/'])
            if result.returncode == 0:
                break
            print(f'  intento {attempt+1} fallo (exit {result.returncode}), reintentando...')
        else:
            raise RuntimeError('No se pudo copiar los patches tras 3 intentos')
    img_dir = local_ch

ch_img_paths = sorted(img_dir.glob('*.npy'))
print(f'Patches: {len(ch_img_paths)}')
assert ch_img_paths, f'No patches found in {img_dir}'

# Manifest real: un nivel arriba de patches/, usa fire_patches/patches (no burned_pct),
# y ya trae transform+crs por tile (evita depender de consulta STAC externa)
manifest_path = cfg['data_dir'].parent / 'manifest.json'
with open(manifest_path) as f:
    manifest = json.load(f)
best_entry = max(manifest['tiles'], key=lambda t: t['fire_patches'] / t['patches'])
best_tile  = best_entry['tile'].lstrip('T')
print(f'Most burned tile: {best_tile} ({best_entry["fire_patches"]}/{best_entry["patches"]} fire patches)')
ch_img_tile = [p for p in ch_img_paths if best_tile in p.stem] or ch_img_paths

ch_ox   = best_entry['transform'][2]
ch_oy   = best_entry['transform'][5]
ch_epsg = int(best_entry['crs'].split(':')[1])

ch_probs_all  = run_inference(ch_img_paths)
tile_indices  = [i for i, p in enumerate(ch_img_paths) if best_tile in p.stem]
ch_probs_tile = [ch_probs_all[i] for i in tile_indices] if tile_indices else ch_probs_all

ch_scene, _, _, _ = reconstruct_scene(ch_img_tile, ch_probs_tile)
print(f'Scene shape: {ch_scene.shape}')

ch_gdf = vectorize_scene(ch_scene, ch_ox, ch_oy, ch_epsg,
                         site_name='chile', event_date=cfg['date'])

if len(ch_gdf) > 0:
    gpkg_local = Path('/content/chile_burn_scar_v22.gpkg') if IN_COLAB else OUTPUTS_V22 / 'chile_burn_scar_v22.gpkg'
    ch_gdf.to_file(gpkg_local, driver='GPKG')
    if IN_COLAB:
        shutil.copy(gpkg_local, OUTPUTS_V22 / 'chile_burn_scar_v22.gpkg')
    print(f'GeoPackage: {OUTPUTS_V22 / "chile_burn_scar_v22.gpkg"}')
else:
    print('No polygons -- GeoPackage skipped')

ch_rgb = reconstruct_rgb_scene(ch_img_tile)
fig_ch = plot_vector_overview(ch_rgb, ch_scene, ch_gdf, cfg['label'], cfg['color'])
save_fig(fig_ch, 'chile_vector_output_v22.png')
print('Chile DONE.')

In [ ]:
# California ZS -- DIFERIDO 2026-07-08. Decision del usuario: esta noche solo
# se procesa Chile, para evitar la crisis de espacio en disco (descargar
# California/Cerrado hubiera requerido otros ~20-40 GB cada uno). El codigo
# queda listo para cuando se agregue este sitio en una sesion futura.
# Ya incluye el fix de estructura de carpetas planas + manifest real
# (fire_patches/patches + transform/crs por tile) descubierto al correr Chile.
# Codigo comentado, no se ejecuta.
"""
# [23b] California ZS #2 (Temperate conifer, Sierra Nevada) -- Dixie Fire 2021
SITE = 'california'
cfg  = SITES[SITE]
print(f'Processing: {cfg["label"]}')
print('=' * 60)

img_dir = cfg['data_dir']
if IN_COLAB:
    local_cf = Path('/content/california_patches')
    if not any(local_cf.glob('*.npy')):
        print('Copying patches from Drive...')
        local_cf.mkdir(exist_ok=True)
        for attempt in range(3):
            result = subprocess.run(['rsync', '-a', f'{cfg["data_dir"]}/', f'{local_cf}/'])
            if result.returncode == 0:
                break
            print(f'  intento {attempt+1} fallo (exit {result.returncode}), reintentando...')
        else:
            raise RuntimeError('No se pudo copiar los patches tras 3 intentos')
    img_dir = local_cf

cf_img_paths = sorted(img_dir.glob('*.npy'))
print(f'Patches: {len(cf_img_paths)}')
assert cf_img_paths, f'No patches found in {img_dir}'

manifest_path = cfg['data_dir'].parent / 'manifest.json'
with open(manifest_path) as f:
    manifest = json.load(f)
best_entry = max(manifest['tiles'], key=lambda t: t['fire_patches'] / t['patches'])
best_tile  = best_entry['tile'].lstrip('T')
print(f'Most burned tile: {best_tile} ({best_entry["fire_patches"]}/{best_entry["patches"]} fire patches)')
cf_img_tile = [p for p in cf_img_paths if best_tile in p.stem] or cf_img_paths

cf_ox   = best_entry['transform'][2]
cf_oy   = best_entry['transform'][5]
cf_epsg = int(best_entry['crs'].split(':')[1])

cf_probs_all  = run_inference(cf_img_paths)
tile_indices  = [i for i, p in enumerate(cf_img_paths) if best_tile in p.stem]
cf_probs_tile = [cf_probs_all[i] for i in tile_indices] if tile_indices else cf_probs_all

cf_scene, _, _, _ = reconstruct_scene(cf_img_tile, cf_probs_tile)
print(f'Scene shape: {cf_scene.shape}')

cf_gdf = vectorize_scene(cf_scene, cf_ox, cf_oy, cf_epsg,
                         site_name='california', event_date=cfg['date'])

if len(cf_gdf) > 0:
    gpkg_local = Path('/content/california_burn_scar_v22.gpkg') if IN_COLAB else OUTPUTS_V22 / 'california_burn_scar_v22.gpkg'
    cf_gdf.to_file(gpkg_local, driver='GPKG')
    if IN_COLAB:
        shutil.copy(gpkg_local, OUTPUTS_V22 / 'california_burn_scar_v22.gpkg')
    print(f'GeoPackage: {OUTPUTS_V22 / "california_burn_scar_v22.gpkg"}')
else:
    print('No polygons -- GeoPackage skipped')

cf_rgb = reconstruct_rgb_scene(cf_img_tile)
fig_cf = plot_vector_overview(cf_rgb, cf_scene, cf_gdf, cfg['label'], cfg['color'])
save_fig(fig_cf, 'california_vector_output_v22.png')
print('California DONE.')
"""

In [ ]:
# Cerrado ZS -- DIFERIDO 2026-07-08. Decision del usuario: esta noche solo
# se procesa Chile, para evitar la crisis de espacio en disco (descargar
# California/Cerrado hubiera requerido otros ~20-40 GB cada uno). El codigo
# queda listo para cuando se agregue este sitio en una sesion futura.
# Ya incluye el fix de estructura de carpetas planas + manifest real
# (fire_patches/patches + transform/crs por tile) descubierto al correr Chile.
# Codigo comentado, no se ejecuta.
"""
# [23c] Cerrado ZS #3 (Tropical dry savanna) -- Mato Grosso wildfires 2023
SITE = 'cerrado'
cfg  = SITES[SITE]
print(f'Processing: {cfg["label"]}')
print('=' * 60)

img_dir = cfg['data_dir']
if IN_COLAB:
    local_ce = Path('/content/cerrado_patches')
    if not any(local_ce.glob('*.npy')):
        print('Copying patches from Drive...')
        local_ce.mkdir(exist_ok=True)
        for attempt in range(3):
            result = subprocess.run(['rsync', '-a', f'{cfg["data_dir"]}/', f'{local_ce}/'])
            if result.returncode == 0:
                break
            print(f'  intento {attempt+1} fallo (exit {result.returncode}), reintentando...')
        else:
            raise RuntimeError('No se pudo copiar los patches tras 3 intentos')
    img_dir = local_ce

ce_img_paths = sorted(img_dir.glob('*.npy'))
print(f'Patches: {len(ce_img_paths)}')
assert ce_img_paths, f'No patches found in {img_dir}'

manifest_path = cfg['data_dir'].parent / 'manifest.json'
with open(manifest_path) as f:
    manifest = json.load(f)
best_entry = max(manifest['tiles'], key=lambda t: t['fire_patches'] / t['patches'])
best_tile  = best_entry['tile'].lstrip('T')
print(f'Most burned tile: {best_tile} ({best_entry["fire_patches"]}/{best_entry["patches"]} fire patches)')
ce_img_tile = [p for p in ce_img_paths if best_tile in p.stem] or ce_img_paths

ce_ox   = best_entry['transform'][2]
ce_oy   = best_entry['transform'][5]
ce_epsg = int(best_entry['crs'].split(':')[1])

ce_probs_all  = run_inference(ce_img_paths)
tile_indices  = [i for i, p in enumerate(ce_img_paths) if best_tile in p.stem]
ce_probs_tile = [ce_probs_all[i] for i in tile_indices] if tile_indices else ce_probs_all

ce_scene, _, _, _ = reconstruct_scene(ce_img_tile, ce_probs_tile)
print(f'Scene shape: {ce_scene.shape}')

ce_gdf = vectorize_scene(ce_scene, ce_ox, ce_oy, ce_epsg,
                         site_name='cerrado', event_date=cfg['date'])

if len(ce_gdf) > 0:
    gpkg_local = Path('/content/cerrado_burn_scar_v22.gpkg') if IN_COLAB else OUTPUTS_V22 / 'cerrado_burn_scar_v22.gpkg'
    ce_gdf.to_file(gpkg_local, driver='GPKG')
    if IN_COLAB:
        shutil.copy(gpkg_local, OUTPUTS_V22 / 'cerrado_burn_scar_v22.gpkg')
    print(f'GeoPackage: {OUTPUTS_V22 / "cerrado_burn_scar_v22.gpkg"}')
else:
    print('No polygons -- GeoPackage skipped')

ce_rgb = reconstruct_rgb_scene(ce_img_tile)
fig_ce = plot_vector_overview(ce_rgb, ce_scene, ce_gdf, cfg['label'], cfg['color'])
save_fig(fig_ce, 'cerrado_vector_output_v22.png')
print('Cerrado DONE.')
"""

In [ ]:
# [24] Cross-site comparison figure -- v2.2 showcase (Chile, mas sitios en el futuro)
results_summary = {
    'Chile ZS #1\n(Mediterranean WUI)': {'gdf': ch_gdf, 'color': SITES['chile']['color']},
    # DIFERIDO -- descomentar cuando se agreguen California y Cerrado:
    # 'California ZS #2\n(Temperate conifer)': {'gdf': cf_gdf, 'color': SITES['california']['color']},
    # 'Cerrado ZS #3\n(Tropical dry savanna)': {'gdf': ce_gdf, 'color': SITES['cerrado']['color']},
}

n_sites = len(results_summary)
fig, axes = plt.subplots(1, n_sites, figsize=(16 * n_sites / 3, 5), facecolor='white')
if n_sites == 1:
    axes = [axes]

for i, (label, r) in enumerate(results_summary.items()):
    gdf      = r['gdf']
    color    = r['color']
    n_poly   = len(gdf)
    total_ha = gdf['area_ha'].sum() if n_poly > 0 else 0

    ax = axes[i]
    if n_poly > 0:
        gdf.plot(ax=ax, color=color, alpha=0.6, edgecolor='white', linewidth=0.3)
    ax.set_title(f'{label}\n{n_poly} polygons -- {total_ha:.0f} ha', fontsize=9)
    ax.set_xlabel('UTM Easting (m)', fontsize=8)
    ax.set_ylabel('UTM Northing (m)', fontsize=8)
    ax.tick_params(labelsize=7)
    for sp in ax.spines.values(): sp.set_color(color)

plt.suptitle(
    'v2.2 -- Burn Scar Perimeters (Zero-Shot)\n'
    'Prithvi-EO-2.0-300M trained on Corrientes + Australia',
    fontsize=11, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'cross_site_vector_summary_v22.png')
print('Cross-site figure saved.')

In [ ]:
# [25] World map -- 3 biomes (2 training + 1 zero-shot: Chile)
from matplotlib.lines import Line2D

TRAIN_SITES = [
    {'name': 'Corrientes\n(Argentina 2022)',     'lat': -29.0, 'lon': -58.0,  'color': '#2563EB'},
    {'name': 'East Gippsland\n(Australia 2020)', 'lat': -37.5, 'lon': 148.0,  'color': '#0891B2'},
]
ZS_SITES = [
    {'name': 'Chile\n(Valparaiso 2023)',      'lat': -33.0,  'lon': -71.5,   'color': '#E74C3C'},
    # DIFERIDO -- descomentar cuando se agreguen California y Cerrado:
    # {'name': 'California\n(Dixie Fire 2021)', 'lat':  40.3,  'lon': -120.85, 'color': '#E67E22'},
    # {'name': 'Cerrado\n(Mato Grosso 2023)',   'lat': -13.5,  'lon':  -51.5,  'color': '#27AE60'},
]

fig, ax = plt.subplots(figsize=(14, 7), facecolor='white')

# World basemap
try:
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    world.plot(ax=ax, color='#E8E8E8', edgecolor='#AAAAAA', linewidth=0.4)
except Exception:
    try:
        import geodatasets
        world = gpd.read_file(geodatasets.get_path('naturalearth.land'))
        world.plot(ax=ax, color='#E8E8E8', edgecolor='#AAAAAA', linewidth=0.4)
    except Exception:
        ax.set_facecolor('#D6EAF8')
        ax.grid(True, alpha=0.3)
        print('World basemap not available -- using plain background')

for s in TRAIN_SITES:
    ax.scatter(s['lon'], s['lat'], s=220, c=s['color'], marker='*',
               zorder=5, edgecolors='white', linewidths=0.8)
    ax.annotate(s['name'], xy=(s['lon'], s['lat']),
                xytext=(s['lon'] + 5, s['lat'] + 5), fontsize=7.5,
                color=s['color'], fontweight='bold',
                arrowprops=dict(arrowstyle='-', color=s['color'], lw=0.8),
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=s['color'], alpha=0.88))

for s in ZS_SITES:
    ax.scatter(s['lon'], s['lat'], s=140, c=s['color'], marker='o',
               zorder=5, edgecolors='white', linewidths=0.8)
    ax.annotate(s['name'], xy=(s['lon'], s['lat']),
                xytext=(s['lon'] + 5, s['lat'] - 7), fontsize=7.5,
                color=s['color'], fontweight='bold',
                arrowprops=dict(arrowstyle='-', color=s['color'], lw=0.8),
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=s['color'], alpha=0.88))

legend_elements = [
    Line2D([0],[0], marker='*', color='w', markerfacecolor='#2563EB',
           markersize=13, label=f'Training biome ({len(TRAIN_SITES)})'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#E74C3C',
           markersize=9,  label=f'Zero-shot site ({len(ZS_SITES)})'),
]
ax.legend(handles=legend_elements, loc='lower left', fontsize=9, frameon=True)
ax.set_title('Wildfire Burn Scar Detection -- Geographic Coverage (v2.2)\n'
             f'{len(TRAIN_SITES)} training biomes (star) + {len(ZS_SITES)} zero-shot biome(s) (circle)',
             fontsize=11, fontweight='bold', pad=10)
ax.set_xlabel('Longitude', fontsize=9)
ax.set_ylabel('Latitude', fontsize=9)
ax.set_xlim(-180, 180)
ax.set_ylim(-65, 80)
ax.grid(alpha=0.25, linestyle='--', linewidth=0.5)
plt.tight_layout()
save_fig(fig, 'world_map_v22.png')
print('World map saved.')

In [ ]:
# [26] Output verification
print('=' * 60)
print('OUTPUT VERIFICATION -- v2.2 (Chile)')
print('=' * 60)

expected = [
    (RESULTS,     'threshold_sweep_v22.png'),
    (RESULTS,     'validation_overview_v22.png'),
    (RESULTS,     'chile_vector_output_v22.png'),
    # DIFERIDO -- descomentar cuando se agreguen California y Cerrado:
    # (RESULTS,     'california_vector_output_v22.png'),
    # (RESULTS,     'cerrado_vector_output_v22.png'),
    (RESULTS,     'cross_site_vector_summary_v22.png'),
    (RESULTS,     'world_map_v22.png'),
    (OUTPUTS_V22, 'chile_burn_scar_v22.gpkg'),
    # (OUTPUTS_V22, 'california_burn_scar_v22.gpkg'),
    # (OUTPUTS_V22, 'cerrado_burn_scar_v22.gpkg'),
]

all_ok = True
for folder, name in expected:
    path   = folder / name
    ok     = path.exists() and path.stat().st_size > 1000
    size   = path.stat().st_size // 1024 if path.exists() else 0
    status = 'OK' if ok else 'MISSING'
    if not ok: all_ok = False
    print(f'  [{status}] {name}  ({size} KB)')

print('=' * 60)
print('ALL OUTPUTS OK' if all_ok else 'WARNING: some outputs missing')

if 'ch_gdf' in dir():
    print()
    print('Burn scar summary:')
    n  = len(ch_gdf)
    ha = ch_gdf['area_ha'].sum() if n > 0 else 0
    print(f'  Chile: {n} polygons -- {ha:.0f} ha')